# LASSO hyperparameter tuning

The diabetes dataset of the LassoBench suite, an $8$-dimensional sparse
regression task. Each evaluation fits a LASSO model, so the objective is
cheap but not free; the run takes a few minutes.

Install LassoBench once, then restart the kernel: Python caches a failed
import, so a package installed after the interpreter started is not picked
up without a restart.

In [ ]:
# import sys
# !{sys.executable} -m pip install git+https://github.com/ksehic/LassoBench.git

In [ ]:
import sys, time, pickle
sys.path.insert(0, '..')          # drop this line if the notebook is at the root

import numpy as np
from joblib import Parallel, delayed

from ibo import (sigma_estimator, get_automatic_kernel_args,
                 evaluate_single_bo, kim1,
                 get_benchmarks, shift_benchmarks,
                 sample_gp_function_, generate_centers,
                 plot_grouped_comparison, plot_mean_ratio_across_functions,
                 report_final_gap, paired_comparison)

import ssl, gc
from tqdm import tqdm

# The suite downloads its datasets over https
ssl._create_default_https_context = ssl._create_unverified_context
import LassoBench

In [ ]:
diabetes_bench = LassoBench.RealBenchmark(pick_data='diabetes',
                                          mf_opt='discrete_fidelity')
d_lasso = diabetes_bench.n_features


def lasso_diabetes_objective(X):
    X = np.atleast_2d(X)
    # BO searches in [0, 1]; LassoBench expects [-1, 1]
    X_scaled = 2 * X - 1
    return np.array([diabetes_bench.evaluate(x) for x in X_scaled]).flatten()


print(f"d = {d_lasso}")

In [ ]:
bounds_lasso = np.array([[0.0, 1.0]] * d_lasso)
d = d_lasso

tree_depth = 3
n_init     = 16 * d       # 40*d would be 320 points before the run starts
n_iter     = 200
n_seeds    = 30
min_leaf   = 16
tau        = 1.5          # splitting threshold, shared across all suites
noise_var  = 1e-8

The length-scale is fitted on an independent Sobol design, as on the other
suites. In dimension eight the spacing of a design of $n$ points is
$n^{-1/8}$, so the fitted value is large relative to the domain and the
estimate is noisier than in dimension two.

In [ ]:
ka = get_automatic_kernel_args({'lasso': lasso_diabetes_objective},
                               d=d, n_points=n_init + n_iter,
                               bounds=(0.0, 1.0), nu=2.5)
kernel_args = ka['lasso']
kernel_args

In [ ]:
configs = [
    # --- EI Group ---
    ("Classic BO - EI", 'fixed', 1,  False, 'ei', {'xi': 0.0}),
    ("IBO_1 - EI",      'fixed', 1,  True,  'ei', {'xi': 0.0}),
    ("IBO_4 - EI",      'fixed', 2,  True,  'ei', {'xi': 0.0}),
    ("IBO_P - EI",      'tree',  2,  True,  'ei', {'xi': 0.0}),

    # --- PI Group ---
    ("Classic BO - PI", 'fixed', 1,  False, 'pi', {'xi': 0.0}),
    ("IBO_1 - PI",      'fixed', 1,  True,  'pi', {'xi': 0.0}),
    ("IBO_4 - PI",      'fixed', 2,  True,  'pi', {'xi': 0.0}),
    ("IBO_P - PI",      'tree',  2,  True,  'pi', {'xi': 0.0}),

    # --- LCB Group ---
    ("Classic BO - LCB", 'fixed', 1, False, 'lcb', {'beta': 1.96}),
    ("IBO_1 - LCB",      'fixed', 1, True,  'lcb', {'beta': 1.96}),
    ("IBO_4 - LCB",      'fixed', 2, True,  'lcb', {'beta': 1.96}),
    ("IBO_P - LCB",      'tree',  2, True,  'lcb', {'beta': 1.96}),

    # --- TS Group ---
    ("Classic BO - TS", 'fixed', 1, False, 'ts', {}),
    ("IBO_1 - TS",      'fixed', 1, True,  'ts', {}),
    ("IBO_4 - TS",      'fixed', 2, True,  'ts', {}),
    ("IBO_P - TS",      'tree',  2, True,  'ts', {}),
]

In [ ]:
histories_lasso = {c[0]: np.zeros((n_seeds, n_iter + 1)) for c in configs}
leaf_lasso = []

start = time.perf_counter()

results = Parallel(n_jobs=4)(
    delayed(evaluate_single_bo)(
        seed, lasso_diabetes_objective, bounds_lasso, configs,
        n_init, d, n_iter,
        kernel_args, noise_var, tree_depth, sigma_estimator,
        tau=tau, n_candidates=256, min_leaf=min_leaf, return_info=True,
    ) for seed in tqdm(range(n_seeds), desc="Lasso diabetes")
)

for seed, seed_histories, infos in results:
    for label, history in seed_histories.items():
        histories_lasso[label][seed, :] = history
    leaf_lasso += [info["n_leaves_hist"].mean()
                   for label, info in infos.items() if label.startswith("IBO_P")]

gc.collect()
print(f"\nExecution time: {time.perf_counter() - start:.1f} s")
print(f"IBO_P mean leaves {np.mean(leaf_lasso):.2f}, "
      f"split in {100*np.mean([l > 1 for l in leaf_lasso]):.0f}% of runs")

In [ ]:
plot_grouped_comparison(histories_lasso, n_seeds=n_seeds,
                        save_path='lasso_diabetes_grouped.pdf')

The objective varies by a fraction of a percent over the whole run, so
report the range before reading anything into the curves.

In [ ]:
allv = np.concatenate([h.ravel() for h in histories_lasso.values()])
print(f"objective range over the experiment: {allv.min():.4f} to {allv.max():.4f} "
      f"({100*(allv.max()-allv.min())/allv.max():.2f}% of the maximum)")

In [ ]:
tag = f"tau{tau}_leaf{min_leaf}_init{n_init}_iter{n_iter}"
with open(f'../results/lasso_{tag}.pkl', 'wb') as fh:
    pickle.dump({'histories': histories_lasso,
                 'leaves': float(np.mean(leaf_lasso)),
                 'kernel_args': kernel_args,
                 'params': {'tau': tau, 'min_leaf': min_leaf, 'n_init': n_init,
                            'n_iter': n_iter, 'n_seeds': n_seeds,
                            'tree_depth': tree_depth, 'd': d}}, fh)
print(f'saved lasso_{tag}.pkl')